In [80]:
import torch
import testdata
# note: had to move this notebook and testdata.py into 
# the multicor_fa directory to run
from _em import _EM_step_no_private_stable, fit_EM_iter

In [81]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [82]:
%autoreload 2

Generate fake data

In [83]:
params = {'d': 15, 'k': [0, 0], 'p': [15, 13], 'n': 5000,'sigsq': [0.3, 0.7]}

Y, W, L, Phi = testdata.simulate_data(params, private_var=False, verbose=True)
W_init, L_init, Phi_init = testdata.initialize_params(W, L, Phi, private_var=False)

# need Y to be N x p_all
Y = Y.T

No private factors, so Y = WZ + E


In [84]:
Sigma_hat = Y.T @ Y / Y.shape[0]

Ground truth

In [85]:
for W_m in W:
    print((W_m @ W_m.T)[:5, :5])

tensor([[15.0905, -0.7598, -4.3597,  3.1973, -1.6555],
        [-0.7598, 16.1165,  2.0864, -1.3946, -0.6938],
        [-4.3597,  2.0864,  9.2589, -1.3581, -0.2284],
        [ 3.1973, -1.3946, -1.3581, 19.6146,  0.7855],
        [-1.6555, -0.6938, -0.2284,  0.7855,  9.3299]])
tensor([[24.4205, -6.8411,  7.6149,  0.2841,  0.4238],
        [-6.8411, 24.9607,  5.3073,  1.4215, -3.6727],
        [ 7.6149,  5.3073, 29.0077, -2.7220,  5.4424],
        [ 0.2841,  1.4215, -2.7220,  7.3141, -2.3024],
        [ 0.4238, -3.6727,  5.4424, -2.3024, 12.8621]])


In [86]:
for Phi_m in Phi:
    print((Phi_m)[:5, :5])

tensor([[0.3000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3000]])
tensor([[0.7000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7000]])


Test EM step with complete data (single case)

In [87]:
W_new, _, Phi_new, _, _ = fit_EM_iter(Y, Sigma_hat, W_init, L_init, Phi_init, maxit = 1000)

In [88]:
for W_m in W_new:
    print((W_m @ W_m.T)[:5, :5])

tensor([[15.2943, -0.5597, -4.7614,  3.0042, -1.2578],
        [-0.5597, 16.7030,  2.1522, -0.7725, -0.9364],
        [-4.7614,  2.1522,  9.5513, -1.0805, -0.5017],
        [ 3.0042, -0.7725, -1.0805, 19.2847,  1.2177],
        [-1.2578, -0.9364, -0.5017,  1.2177,  9.0521]])
tensor([[24.2403, -6.9636,  7.3036,  0.4564,  0.3271],
        [-6.9636, 24.7629,  4.7618,  1.5769, -3.1629],
        [ 7.3036,  4.7618, 29.4055, -2.8053,  6.1981],
        [ 0.4564,  1.5769, -2.8053,  7.0183, -2.1865],
        [ 0.3271, -3.1629,  6.1981, -2.1865, 12.7580]])


In [89]:
for Phi_m in Phi_new:
    print((Phi_m)[:5, :5])

tensor([[0.3202, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3267, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2887, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3072, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.2844]])
tensor([[0.7661, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7214, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7507, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.6880, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.6940]])


Test EM step with missing data (single case)

In [90]:
Y_miss = Y.clone().detach()

In [91]:
Y_miss[1000:1025, :params['p'][0]] = float('nan')
Y_miss[1025:1075, params['p'][0]:] = float('nan')

In [92]:
Y_miss[1020:1030, params['p'][0]-3:params['p'][0]+3]

tensor([[     nan,      nan,      nan,   4.8639,  -6.7614,   0.3358],
        [     nan,      nan,      nan,  17.9496, -14.8280,   8.0759],
        [     nan,      nan,      nan,   2.9384,   1.4298,  -0.0889],
        [     nan,      nan,      nan,   1.3393,   3.9662,  -0.9686],
        [     nan,      nan,      nan,  -5.6225,   0.5999,  -2.9480],
        [  3.0134,  -2.9063,   7.5988,      nan,      nan,      nan],
        [  5.1627,  -3.8869,  -7.0494,      nan,      nan,      nan],
        [  1.6543,  -4.1427,   1.0610,      nan,      nan,      nan],
        [  0.2535,  -1.5242,  -0.3070,      nan,      nan,      nan],
        [  4.2765,   4.7316,   3.6049,      nan,      nan,      nan]])

In [93]:
Sigma_hat = Y_miss.nan_to_num().T @ Y_miss.nan_to_num() / Y_miss.shape[0]

In [94]:
W_new, _, Phi_new, _, _ = fit_EM_iter(Y_miss, Sigma_hat, W_init, L_init, Phi_init, maxit = 1000, impute_modes = True)

In [95]:
for W_m in W_new:
    print((W_m @ W_m.T)[:5, :5])

tensor([[15.1874, -0.5725, -4.7224,  2.9665, -1.2679],
        [-0.5725, 16.5604,  2.1543, -0.7924, -0.8921],
        [-4.7224,  2.1543,  9.4462, -1.0638, -0.4737],
        [ 2.9665, -0.7924, -1.0638, 19.1443,  1.1744],
        [-1.2679, -0.8921, -0.4737,  1.1744,  9.0334]])
tensor([[24.0809, -6.9269,  7.2226,  0.4744,  0.3195],
        [-6.9269, 24.2131,  4.6868,  1.4901, -3.1109],
        [ 7.2226,  4.6868, 29.0813, -2.7940,  6.1549],
        [ 0.4744,  1.4901, -2.7940,  6.9063, -2.1751],
        [ 0.3195, -3.1109,  6.1549, -2.1751, 12.5866]])


In [96]:
for Phi_m in Phi_new:
    print(Phi_m[:5, :5])

tensor([[0.3510, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3561, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3332, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3265, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.2963]])
tensor([[0.6923, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.9561, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.8014, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7295, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7280]])


Attempt at more systematic testing for complete data case

In [109]:
metrics = {
    'WWt_mse': [],
    'WWt_corr': [],
    'Phi_mse': [],
    'Phi_corr': [],
}
# simulate 1000 runs 
for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]
    
    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_mse'].append(torch.mean((WW_test - WW)**2).item())
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    metrics['Phi_mse'].append(torch.mean((P_test - P)**2).item())
    metrics['Phi_corr'].append(torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item())

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


Manual note: 1000 simulations with missing data took ~ 1m 8.6s to run

In [110]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_mse
	Mean: 0.0972
	Min: 0.0368
	Max: 0.319
WWt_corr
	Mean: 0.9981
	Min: 0.996
	Max: 0.9991
Phi_mse
	Mean: 0.0007
	Min: 0.0002
	Max: 0.0047
Phi_corr
	Mean: 0.9918
	Min: 0.9388
	Max: 0.9978


Attempt at more systematic testing for missing data case

In [117]:
metrics = {
    'WWt_mse': [],
    'WWt_corr': [],
    'Phi_mse': [],
    'Phi_corr': [],
}
# simulate 1000 runs 
for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    # insert missing data
    Y[1000:1025, :params['p'][0]] = float('nan')
    Y[1025:1075, params['p'][0]:] = float('nan')
    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    # need to fill in NA Sigma values here
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute_modes=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_mse'].append(torch.mean((WW_test - WW)**2).item())
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    metrics['Phi_mse'].append(torch.mean((P_test - P)**2).item())
    metrics['Phi_corr'].append(torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item())

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


Manual note: 1000 simulations with missing data took ~ 1m 18.7s

In [118]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_mse
	Mean: 0.0907
	Min: 0.0289
	Max: 0.3056
WWt_corr
	Mean: 0.9982
	Min: 0.9963
	Max: 0.9992
Phi_mse
	Mean: 0.0078
	Min: 0.0027
	Max: 0.0258
Phi_corr
	Mean: 0.9592
	Min: 0.8819
	Max: 0.9901
